# Module 6: Building the 1D-CNN in PyTorch

---

This module defines the full ML pipeline that will consume the 1M-scan dataset from Module 5 and train a 1D-CNN to predict methane mole fraction (χ) from a calibrated absorbance spectrum + thermodynamic conditions (T, P).

**Learning Objectives:**
- Build a custom PyTorch `Dataset` class that reads from HDF5 and applies normalization on-the-fly
- Define the 1D-CNN architecture in PyTorch (`nn.Module`)
- Verify the end-to-end forward pass on your target device
- Benchmark throughput on CPU (Mac) vs NVIDIA GPU to motivate the GPU training workflow

**Deliverable:** A verified PyTorch `MethaneCNN` model and `MethaneDataset` DataLoader ready for training in Module 7.

**Dataset:** `dataset_1M-2026-02-20.h5`

---

---
## 6.1 — Environment & Device Setup

We auto-detect the best available device in priority order: **CUDA → MPS → CPU**.

- **CUDA** — NVIDIA GPU (your GPU laptop, accessed via SSH)
- **MPS** — Apple Metal Performance Shaders (MacBook GPU acceleration)
- **CPU** — fallback, used for the Mac baseline benchmark

> ⚠️ If you are running on the Mac and see `device = mps`, PyTorch will use the Apple GPU for the forward pass but training throughput will be lower than NVIDIA CUDA. We use `cpu` explicitly for the Mac *CPU* benchmark in §6.5.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

import h5py
import numpy as np
import time
import os
from pathlib import Path

# ── Device selection ────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"PyTorch version : {torch.__version__}")
print(f"Device selected : {device}")
if device.type == "cuda":
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# ── Dataset path ────────────────────────────────────────────
# Adjust this path if running on a different machine (e.g. your NVIDIA laptop)
DATASET_PATH = Path.home() / "RedwoodLabs/methane-ml-course/data/datasets/dataset_1M-2026-02-20.h5"

assert DATASET_PATH.exists(), f"Dataset not found at {DATASET_PATH}"
print(f"Dataset found   : {DATASET_PATH}")
print(f"File size       : {DATASET_PATH.stat().st_size / 1e9:.2f} GB")


In [ ]:
# ── Inspect HDF5 structure & normalization attributes ────────
with h5py.File(DATASET_PATH, "r") as hf:
    N            = hf["a"].shape[0]
    NUM_PTS      = hf["a"].shape[1]
    a_global_max  = float(hf.attrs["a_global_max"])
    T_global_max  = float(hf.attrs["T_global_max"])
    P_global_max  = float(hf.attrs["P_global_max"])
    xi_global_max = float(hf.attrs["xi_global_max"])
    nu_grid       = hf["nu"][:]

print(f"Total scans     : {N:,}")
print(f"Spectral points : {NUM_PTS}")
print(f"ν range         : {nu_grid[0]:.2f} – {nu_grid[-1]:.2f} cm⁻¹")
print()
print("Normalization constants (stored in HDF5 attrs):")
print(f"  α global max  : {a_global_max:.6f}")
print(f"  T global max  : {T_global_max:.1f} K")
print(f"  P global max  : {P_global_max:.4f} atm")
print(f"  χ global max  : {xi_global_max*1e6:.2f} ppm")


---
## 6.2 — Custom Dataset & DataLoader

### 6.2.1 — MethaneDataset

The `MethaneDataset` class:
- Opens the HDF5 file **once** and keeps it open (lazy, memory-mapped reads — no full dataset loaded into RAM)
- Returns a normalized `(3, NUM_PTS)` input tensor and a scalar normalized target for each scan

Normalization applied on-the-fly (from Module 5's max-only convention):

| Channel | Content | Formula |
|---------|---------|---------|
| 0 | Absorbance spectrum | `α_norm[k] = α_scan[k] / α_global_max` |
| 1 | Temperature (broadcast) | `T_norm = T_scan / T_global_max` |
| 2 | Pressure (broadcast) | `P_norm = P_scan / P_global_max` |
| Target | Mole fraction | `χ_norm = χ_scan / χ_global_max` |


In [ ]:
class MethaneDataset(Dataset):
    """
    Lazy HDF5 dataset for methane CNN training.

    Reads single samples from disk on demand — the full 1M-scan dataset
    (~112 GB) never needs to fit in RAM.

    Returns
    -------
    x : torch.FloatTensor, shape (3, NUM_PTS)
        Channel 0 — normalised absorbance spectrum (element-wise)
        Channel 1 — T_norm broadcast to all spectral points
        Channel 2 — P_norm broadcast to all spectral points
    y : torch.FloatTensor, scalar
        Normalised mole fraction (χ_scan / χ_global_max)
    """

    def __init__(self, hdf5_path: str | Path):
        self.path = str(hdf5_path)

        # Read normalization constants and dataset size once at init
        with h5py.File(self.path, "r") as hf:
            self.N            = hf["a"].shape[0]
            self.num_pts      = hf["a"].shape[1]
            self.a_global_max  = float(hf.attrs["a_global_max"])
            self.T_global_max  = float(hf.attrs["T_global_max"])
            self.P_global_max  = float(hf.attrs["P_global_max"])
            self.xi_global_max = float(hf.attrs["xi_global_max"])

        # HDF5 file handle opened lazily per worker (see __getitem__)
        self._hf = None

    def _open(self):
        """Open the HDF5 file handle (once per DataLoader worker process)."""
        if self._hf is None:
            self._hf = h5py.File(self.path, "r")

    def __len__(self):
        return self.N

    def __getitem__(self, idx):
        self._open()

        # ── Read raw values ──────────────────────────────────
        a_scan  = self._hf["a"][idx]          # (NUM_PTS,)  float32
        T_scan  = float(self._hf["T"][idx])
        P_scan  = float(self._hf["P"][idx])
        xi_scan = float(self._hf["xi"][idx])

        # ── Normalise ────────────────────────────────────────
        a_norm  = a_scan / self._hf.attrs["a_global_max"]    # element-wise
        T_norm  = T_scan  / self.T_global_max
        P_norm  = P_scan  / self.P_global_max
        xi_norm = xi_scan / self.xi_global_max

        # ── Build (3, NUM_PTS) input tensor ──────────────────
        T_channel = np.full(self.num_pts, T_norm, dtype=np.float32)
        P_channel = np.full(self.num_pts, P_norm, dtype=np.float32)

        x = torch.from_numpy(
            np.stack([a_norm.astype(np.float32), T_channel, P_channel])
        )                                       # (3, NUM_PTS)
        y = torch.tensor(xi_norm, dtype=torch.float32)

        return x, y


print("✅ MethaneDataset defined")


### 6.2.2 — Train / Validation / Test Split

We use a **70 / 15 / 15** split:

| Split | Fraction | Scans (1M dataset) |
|-------|----------|-------------------|
| Train | 70% | ~700,000 |
| Val   | 15% | ~150,000 |
| Test  | 15% | ~150,000 |

The split is performed once with a fixed random seed for reproducibility.


In [ ]:
SEED       = 42
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15
# TEST_FRAC is the remainder (0.15)

full_dataset = MethaneDataset(DATASET_PATH)
N_total = len(full_dataset)

n_train = int(TRAIN_FRAC * N_total)
n_val   = int(VAL_FRAC   * N_total)
n_test  = N_total - n_train - n_val

train_ds, val_ds, test_ds = random_split(
    full_dataset,
    [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED),
)

print(f"Total  : {N_total:>10,}")
print(f"Train  : {n_train:>10,}  ({n_train/N_total*100:.1f}%)")
print(f"Val    : {n_val:>10,}  ({n_val  /N_total*100:.1f}%)")
print(f"Test   : {n_test:>10,}  ({n_test /N_total*100:.1f}%)")


### 6.2.3 — DataLoader Configuration

`num_workers` and `pin_memory` are tuned per device:

| Device | `num_workers` | `pin_memory` | Notes |
|--------|--------------|-------------|-------|
| CUDA   | 4            | True        | Async CPU→GPU transfer |
| MPS    | 0            | False       | MPS doesn't support multiprocessing well |
| CPU    | 2            | False       | Light parallelism for prefetch |


In [ ]:
BATCH_SIZE = 256

# ── DataLoader config per device ─────────────────────────────
if device.type == "cuda":
    loader_kwargs = dict(num_workers=4, pin_memory=True, persistent_workers=True)
elif device.type == "mps":
    loader_kwargs = dict(num_workers=0, pin_memory=False)
else:  # cpu
    loader_kwargs = dict(num_workers=2, pin_memory=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  **loader_kwargs)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

print(f"Batch size      : {BATCH_SIZE}")
print(f"Train batches   : {len(train_loader):,}")
print(f"Val batches     : {len(val_loader):,}")
print(f"Test batches    : {len(test_loader):,}")
print(f"num_workers     : {loader_kwargs['num_workers']}")
print(f"pin_memory      : {loader_kwargs.get('pin_memory', False)}")


In [ ]:
# ── Sanity-check a single batch ─────────────────────────────
x_batch, y_batch = next(iter(train_loader))

print(f"x_batch shape   : {tuple(x_batch.shape)}  (batch, channels, spectral_pts)")
print(f"y_batch shape   : {tuple(y_batch.shape)}  (batch,)")
print()
print(f"x dtype         : {x_batch.dtype}")
print(f"x min/max       : [{x_batch.min():.4f}, {x_batch.max():.4f}]")
print(f"y dtype         : {y_batch.dtype}")
print(f"y min/max       : [{y_batch.min():.4f}, {y_batch.max():.4f}]")
print()
print("✅ DataLoader returns correctly shaped, normalised tensors")


---
## 6.3 — Model Architecture

### Design

The `MethaneCNN` follows the architecture laid out in the course outline, adapted to PyTorch's `(batch, channels, length)` convention:

```
Input : (batch, 3, NUM_PTS)
          │
          ▼
Conv1d(3 → 32, kernel=5, padding=2)  ← 'same' padding: output length = NUM_PTS
ReLU
          │
          ▼
Conv1d(32 → 64, kernel=5, padding=2)
ReLU
Dropout(p=0.2)
          │
          ▼
Flatten  → (batch, 64 × NUM_PTS)
          │
          ▼
Linear(64 × NUM_PTS → 128)
ReLU
          │
          ▼
Linear(128 → 1)
          │
          ▼
Output : (batch,)   ← χ_norm (scalar, normalised mole fraction)
```

`padding=2` with `kernel_size=5` is the PyTorch equivalent of `padding='same'` for odd kernels, keeping the spectral length constant through both conv layers.

### Why 1D convolutions?
The spectral dimension is analogous to time in a 1D signal — nearby wavenumber points are strongly correlated (peak shape, shoulders). Conv1D extracts these local line-shape features with weight sharing, giving the model translation equivariance along the spectral axis.


In [ ]:
class MethaneCNN(nn.Module):
    """
    1D-CNN for predicting normalised methane mole fraction (χ_norm)
    from a 3-channel spectral input tensor.

    Parameters
    ----------
    num_spec_points : int
        Number of spectral points in each scan (from HDF5 shape).
    dropout_p : float
        Dropout probability after the second conv block (default 0.2).
    """

    def __init__(self, num_spec_points: int, dropout_p: float = 0.2):
        super().__init__()
        self.num_spec_points = num_spec_points

        # ── Convolutional feature extractor ───────────────────
        # padding=2 with kernel_size=5 keeps output length = input length
        self.conv_stack = nn.Sequential(
            nn.Conv1d(in_channels=3,  out_channels=32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
        )

        # ── Regression head ───────────────────────────────────
        flat_size = 64 * num_spec_points
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_size, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        x : (batch, 3, num_spec_points)

        Returns
        -------
        (batch,) — predicted χ_norm
        """
        x = self.conv_stack(x)           # (batch, 64, num_spec_points)
        x = self.head(x)                 # (batch, 1)
        return x.squeeze(-1)             # (batch,)


print("✅ MethaneCNN defined")


In [ ]:
# ── Instantiate and inspect ──────────────────────────────────
model = MethaneCNN(num_spec_points=NUM_PTS)

# Parameter count
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Spectral points : {NUM_PTS}")
print(f"Flat size       : {64 * NUM_PTS:,}  (64 channels × {NUM_PTS} points)")
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print()
print(model)


---
## 6.4 — Forward Pass Verification

Move the model and a sample batch to the target device and confirm:
1. Tensor shapes are correct at each stage
2. Output is in the normalised range [0, 1]
3. Backward pass (gradient computation) runs without error


In [ ]:
# ── Move model to device ─────────────────────────────────────
model = model.to(device)
print(f"Model on device : {next(model.parameters()).device}")

# ── Forward pass with one batch ──────────────────────────────
x_dev = x_batch.to(device)
y_dev = y_batch.to(device)

model.eval()
with torch.no_grad():
    y_pred = model(x_dev)

print(f"\nInput  shape    : {tuple(x_dev.shape)}")
print(f"Output shape    : {tuple(y_pred.shape)}")
print(f"y_pred range    : [{y_pred.min().item():.4f}, {y_pred.max().item():.4f}]")
print(f"y_true range    : [{y_dev.min().item():.4f},  {y_dev.max().item():.4f}]")
print("\n✅ Forward pass successful")


In [ ]:
# ── Backward pass (gradient) check ──────────────────────────
model.train()
loss_fn   = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

optimizer.zero_grad()
y_pred = model(x_dev)
loss   = loss_fn(y_pred, y_dev)
loss.backward()
optimizer.step()

print(f"Initial loss    : {loss.item():.6f}")
print(f"  (untrained model — expect ~0.1–0.3 on normalised χ)")

# Verify gradients flowed
grad_norms = [p.grad.norm().item() for p in model.parameters() if p.grad is not None]
print(f"Gradient norms  : min={min(grad_norms):.2e}  max={max(grad_norms):.2e}")
print("\n✅ Backward pass successful — gradients computed for all layers")


---
## 6.5 — CPU vs GPU Benchmark

We time **forward + backward pass throughput** over a fixed number of batches to compare:

- **Mac CPU** — run this notebook locally on your MacBook
- **NVIDIA GPU** — SSH into your GPU laptop, copy this notebook there, and run it again

The benchmark uses the same `BATCH_SIZE` and `N_BATCHES` on both machines so results are directly comparable.

> 💡 **How to run on the NVIDIA laptop:**
> ```bash
> # From your Mac — copy the notebook over
> scp ~/RedwoodLabs/methane-ml-course/notebooks/Module_06_Build_1D_CNN.ipynb \
>     <user>@<GPU_LAPTOP_IP>:~/methane-ml-course/notebooks/
>
> # Also copy the dataset if it's not already there
> rsync -ah --progress \
>     ~/RedwoodLabs/methane-ml-course/data/datasets/dataset_1M-2026-02-20.h5 \
>     <user>@<GPU_LAPTOP_IP>:~/methane-ml-course/data/datasets/
>
> # SSH in and launch Jupyter
> ssh <user>@<GPU_LAPTOP_IP>
> jupyter lab --no-browser --port=8888
>
> # Then open in your Mac browser via SSH tunnel:
> ssh -L 8888:localhost:8888 <user>@<GPU_LAPTOP_IP>
> ```


In [ ]:
def throughput_benchmark(model, loader, device, n_batches=100, label=""):
    """
    Measure forward+backward pass throughput over n_batches batches.
    Returns samples/second.
    """
    loss_fn   = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    model.train()

    # Warm-up (2 batches — not timed)
    for i, (x, y) in enumerate(loader):
        if i >= 2:
            break
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss_fn(model(x), y).backward()
        optimizer.step()

    # Synchronise GPU before timing
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()

    t0 = time.perf_counter()

    samples_seen = 0
    for i, (x, y) in enumerate(loader):
        if i >= n_batches:
            break
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss_fn(model(x), y).backward()
        optimizer.step()
        samples_seen += x.size(0)

    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()

    elapsed = time.perf_counter() - t0
    throughput = samples_seen / elapsed

    print(f"[{label}]")
    print(f"  Device          : {device}")
    print(f"  Batches timed   : {n_batches}  ×  {BATCH_SIZE} samples")
    print(f"  Elapsed         : {elapsed:.2f} s")
    print(f"  Throughput      : {throughput:,.0f} samples/s")
    batches_per_s = n_batches / elapsed
    samples_per_epoch = n_train
    est_epoch_s = samples_per_epoch / throughput
    print(f"  Est. time/epoch : {est_epoch_s/60:.1f} min  ({samples_per_epoch:,} train samples)")
    return throughput


N_BENCH_BATCHES = 100
print(f"Running {N_BENCH_BATCHES}-batch benchmark on {device}...")
print()
throughput = throughput_benchmark(
    model, train_loader, device, n_batches=N_BENCH_BATCHES, label=str(device).upper()
)


### Results Table

Fill this in after running the benchmark on both machines:

| Machine | Device | Throughput (samples/s) | Est. time / epoch |
|---------|--------|----------------------|-------------------|
| MacBook Pro | CPU | ___ | ___ min |
| GPU Laptop | CUDA (___) | ___ | ___ min |

> **Interpreting results:** A modern NVIDIA GPU (e.g. RTX 3080/4080) typically delivers 10–30× higher throughput than a Mac CPU on this workload, reducing a multi-hour training run to minutes.

---


---
## Summary

### What We Built

1. **`MethaneDataset`** — Lazy HDF5 reader with on-the-fly max-only normalization; returns `(3, NUM_PTS)` tensors without loading the full dataset into RAM
2. **Train / Val / Test split** — 70/15/15 with fixed seed for reproducibility
3. **`DataLoader`** — Device-aware configuration (`num_workers`, `pin_memory`)
4. **`MethaneCNN`** — Two Conv1d blocks + dropout + two-layer regression head; verified forward and backward pass on target device
5. **Throughput benchmark** — Baseline for CPU vs NVIDIA GPU comparison

### Up Next — Module 7: Training

- Full training loop with Adam + MSE loss
- Per-epoch train/val loss logging
- Live-updating loss curves in Jupyter
- Model checkpointing (`best_model.pt`)
- Estimated training time on GPU vs CPU
